In [0]:
data = [
 ("James,,Smith",["Java","Scala","C++"],["Spark","Java"],"OH","CA"),
 ("Michael,Rose,",["Spark","Java","C++"],["Spark","Java"],"NY","NJ"),
 ("Robert,,Williams",["CSharp","VB"],["Spark","Python"],"UT","NV")
]

from pyspark.sql.types import StringType, ArrayType,StructType,StructField
schema = StructType([ 
    StructField("name",StringType(),True), 
    StructField("languagesAtSchool",ArrayType(StringType()),True), 
    StructField("languagesAtWork",ArrayType(StringType()),True), 
    StructField("currentState", StringType(), True), 
    StructField("previousState", StringType(), True)
  ])

df = spark.createDataFrame(data=data,schema=schema)
df.printSchema()
df.show()

In [0]:
from pyspark.sql import functions as F

df_exploded_work = df.withColumn("exploded_language_at_work", F.explode(F.col("languagesAtWork")))
display(df_exploded_work)

In [0]:
from pyspark.sql import functions as F

# 1. Get array length
df_length = df.withColumn("school_length", F.size(F.col("languagesAtSchool")))
display(df_length)

# 2. Check if array contains a value
df_contains = df_length.withColumn("has_Java", F.array_contains(F.col("languagesAtSchool"), "Java"))
display(df_contains)

# 3. Explode array to rows
df_exploded = df_contains.withColumn("exploded_language", F.explode(F.col("languagesAtSchool")))
display(df_exploded)



In [0]:
# 4. Split string column to array
df_split = df_contains.withColumn("nameAsArray", F.split(F.col("name"), ","))
display(df_split)



In [0]:

# 5. Merge columns into array
df_array = df_contains.withColumn("state_array", F.array(F.col("currentState"), F.col("previousState")))
display(df_array)


In [0]:
# 6. Array distinct
# This step creates a new column 'distinct_school' containing only the unique values from the 'languagesAtSchool' array for each row.
df_distinct = df_contains.withColumn("distinct_school", F.array_distinct(F.col("languagesAtSchool")))
display(df_distinct)

In [0]:

# 7. Array union - combine two arrays with distinct records 
df_union = df_contains.withColumn("school_union_work", F.array_union(F.col("languagesAtSchool"), F.col("languagesAtWork")))
display(df_union)

In [0]:
from pyspark.sql import functions as F

# add any value to all the cells of the array in another column
df5 = df.withColumn("arr_appended", F.array_append(F.col("languagesAtSchool"), "Java"))
display(df5)

# add elements in the first of the list
df6 = df.withColumn("arr_prepended", F.array_prepend(F.col("languagesAtSchool"), "C++"))
display(df6)

df7 = df.withColumn("arr_remove", F.array_remove(F.col("languagesAtSchool"), "Java"))
display(df7) # we cannot remove more than one element at a time

# remove multiple values in list at a time using column name remove_multiple_elements
remove_values = ["Java", "C++"]
df8 = df.withColumn(
    "remove_multiple_elements",
    F.array_remove(F.array_remove(F.col("languagesAtSchool"), remove_values[0]), remove_values[1])
)
display(df8)

df9 = df.withColumn("allcourses",F.concat("languagesAtSchool","languagesAtWork"))
display(df9)

# convert all the elements in the column to upper case the column has_java
df10 = df.withColumn("upper", F.array(F.upper(F.col("name"))))
display(df10)

df11 = df.withColumn("lower", F.array(F.lower(F.col("previousState"))))
display(df11)

df12 = df.withColumn("lower_concat", F.lower(F.concat(F.col("currentState"), F.col("previousState"))))
display(df12)

In [0]:
# 8. Array intersection
# This step creates a new column 'school_intersect_work' containing the common elements between
# the 'languagesAtSchool' and 'languagesAtWork' arrays for each row. Only values present in both arrays are included.
df_intersect = df_contains.withColumn("school_intersect_work", F.array_intersect(F.col("languagesAtSchool"), F.col("languagesAtWork")))
display(df_intersect)

# 9. Array except
# This step creates a new column 'school_except_work' containing the elements from the 'languagesAtSchool' array
# that are not present in the 'languagesAtWork' array for each row. It effectively subtracts the second array from the first.
df_except = df_contains.withColumn("school_except_work", F.array_except(F.col("languagesAtSchool"), F.col("languagesAtWork")))
display(df_except)

In [0]:
# 10. Array position
# tell the position of the element in the array we specify
df_position = df_contains.withColumn("java_position", F.array_position(F.col("languagesAtSchool"), "Java"))
display(df_position)

# 11. Array reverse
# reverse the elements in the array
df_reverse = df_contains.withColumn("school_reverse", F.reverse(F.col("languagesAtSchool")))
display(df_reverse)

# 12. Array sort
#sort the selected columns in ascending order
df_sort = df_contains.withColumn("school_sorted", F.array_sort(F.col("languagesAtSchool")))
display(df_sort)

In [0]:
from pyspark.sql import functions as F

# 1. Convert DataFrame to JSON string column
df_json = df.withColumn("json_str", F.to_json(F.struct([df[x] for x in df.columns])))
display(df_json)

# 2. Parse JSON string column to struct
df_with_json = df_json.withColumn("parsed_json", F.from_json(F.col("json_str"), df.schema))
display(df_with_json)

# 3. Extract field from JSON string
df_field = df_json.withColumn("name_from_json", F.get_json_object(F.col("json_str"), "$.name"))
display(df_field)


# 5. Explode JSON array field
df_explode_json = df_from_json.withColumn("school_language", F.explode(F.col("languagesAtSchool")))
display(df_explode_json)


In [0]:


from pyspark.sql.functions import current_timestamp

source_data = [
    (1, "Alice", "NY"),
    (2, "Bob", "LA"),
    (3, "Charlie", "SF"),(4, "sandeep", "BH")
]
source_df = spark.createDataFrame(source_data, ["eid", "ename", "ecity"]) \
    .withColumn("withstamp", current_timestamp()) \
    .withColumn("updated_time", current_timestamp())

# Create target table with initial data
target_data = [
    (1, "Alice", "CA"),
    (2, "Bob", "LA"),(3, "sai", "IN")
]
target_df = spark.createDataFrame(target_data, ["eid", "ename", "ecity"]) \
    .withColumn("withstamp", current_timestamp()) \
    .withColumn("updated_time", current_timestamp())

# Register as temp views for SQL MERGE
source_df.createOrReplaceTempView("source_table")
target_df.createOrReplaceTempView("target_table")

In [0]:
%sql
select * from source_table;

In [0]:
%sql
select * from target_table;

In [0]:
%sql
CREATE OR REPLACE TABLE sandeep.source_table AS
SELECT * FROM source_table;

CREATE OR REPLACE TABLE sandeep.target_table AS
SELECT * FROM target_table;

In [0]:
%sql
MERGE INTO workspace.sandeep.source_table AS target
USING sandeep.target_table AS source
ON target.eid = source.eid
WHEN MATCHED THEN
  UPDATE SET target.ename = source.ename,
             target.ecity = source.ecity,
             target.updated_time = now()
WHEN NOT MATCHED THEN
  INSERT (eid, ename, ecity, withstamp, updated_time)
  VALUES (source.eid, source.ename, source.ecity, source.withstamp,now())

In [0]:
%sql
select * from workspace.sandeep.source_table

In [0]:
%sql
select * from workspace.sandeep.target_table

In [0]:
data = [("abc123",), ("456def",), ("hello789",), ("321world",), ("test456",),("test",)]
df_mixed = spark.createDataFrame(data, ["mixed_col"])
display(df_mixed)

In [0]:
from pyspark.sql.functions import udf
import re
from pyspark.sql.types import StringType

def extract_numbers(s):
    if not s or not re.search(r'[A-Za-z]', s):
        return None
    nums = re.findall(r'\d+', s)
    return ''.join(nums) if nums else None

extract_numbers_udf = udf(extract_numbers, StringType()) # we register the python function as udf 


df_numbers = df_mixed.withColumn("numbers_extracted", extract_numbers_udf("mixed_col"))
display(df_numbers)

In [0]:
from pyspark.sql.functions import udf
import re
from pyspark.sql.types import StringType

def extract_numbers(s):
    if s is None or not any(c.isalpha() for c in s):
        return None
    nums = re.findall(r'\d+', s)
    return ''.join(nums) if nums else None

extract_numbers_udf1 = udf(extract_numbers, StringType())

df_numbers = df_mixed.withColumn("numbers_extracted", extract_numbers_udf1("mixed_col"))
display(df_numbers)